# 06. 결론 — 4개 집단 비교 및 표준화 위험 점수

> 이 노트북은 원본 통합 노트북 `소스_코드_8팀_생체_신호_기반_흡연_여부_비교_시각화_프로젝트__-_종합.ipynb` 의 관련 셀들을 주제별로 재구성한 것입니다. 코드는 원본 그대로 옮겨왔으며(실행 결과만 초기화), 새로 작성하거나 수정한 로직은 없습니다.
>
> 실행하려면 `train_dataset.csv`(Kaggle: Smoker Status Prediction Dataset)를 동일 폴더에 두어야 합니다. 데이터 파일은 저장소에 포함되어 있지 않습니다.
>
> 원본에는 서로 다른 시점에 작성된 3개의 전처리 버전이 섞여 있습니다 (자세한 내용은 `docs/03_issues_and_troubleshooting.md` 참고). 이 재구성본은 그 버전들을 **삭제하거나 하나로 합치지 않고**, 각 노트북 안에서 '버전 A/B/C'로 구분해 모두 보존했습니다.

저BMI/고BMI × 비흡연/흡연 4개 집단으로 나누어 표준화 위험 점수를 계산하고, 타겟 집단(고BMI 흡연자)을 선정하는 최종 분석입니다.

## 4개 집단 비중 및 평균 지표

In [ ]:
#결론
#파이차트_개입 타겟 집단 비중
target_count = train_clean.groupby(['high_BMI','smoking']).size()
labels = ['저BMI-비흡연', '저BMI-흡연', '고BMI-비흡연', '고BMI-흡연']
colors = [COLOR_LOW_BMI, COLOR_SMOKER, COLOR_HIGH_BMI, COLOR_SMOKER]

plt.figure(figsize=(6,6))
plt.pie(target_count.values, labels=labels, autopct='%1.1f%%', colors=colors)
plt.title('BMI × 흡연 조합별 집단 비중', color=COLOR_TEXT)
plt.tight_layout()
plt.show()

In [ ]:
#결론
#고BMI 흡연자 위험 강조 그래프
target_vars = ['ALT', 'Gtp', 'HDL', 'waist(cm)']
target_mean = train_clean.groupby(['high_BMI','smoking'])[target_vars].mean()

display(target_mean)

## 4개 집단 표준화 위험 점수 계산

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

COLOR_SMOKER = '#A63A50'       # 흡연자 강조
COLOR_HIGH_BMI = '#C65D6E'     # 고BMI 강조
COLOR_NONSMOKER = '#6B7075'    # 비흡연자
COLOR_LOW_BMI = '#2F3437'      # 저BMI
COLOR_TEXT = '#1E1E1E'
COLOR_GRID = '#D9D9D9'

plt.rcParams['axes.unicode_minus'] = False

train_clean = train[
    (train['Gtp'] < 500) &
    (train['ALT'] < 500) &
    (train['AST'] < 500) &
    (train['LDL'] < 400) &
    (train['serum creatinine'] < 3.0) &
    (train['eyesight(left)'] <= 3.0) &
    (train['eyesight(right)'] <= 3.0)
].copy()


train_clean['BMI'] = train_clean['weight(kg)'] / ((train_clean['height(cm)'] / 100) ** 2)
train_clean['high_BMI'] = (train_clean['BMI'] >= 25).astype(int)

def make_group(row):
    if row['high_BMI'] == 0 and row['smoking'] == 0:
        return '저BMI-비흡연'
    elif row['high_BMI'] == 0 and row['smoking'] == 1:
        return '저BMI-흡연'
    elif row['high_BMI'] == 1 and row['smoking'] == 0:
        return '고BMI-비흡연'
    else:
        return '고BMI-흡연'

train_clean['group4'] = train_clean.apply(make_group, axis=1)

group_order = ['저BMI-비흡연', '저BMI-흡연', '고BMI-비흡연', '고BMI-흡연']
group_colors = [COLOR_LOW_BMI, COLOR_NONSMOKER, COLOR_HIGH_BMI, COLOR_SMOKER]

In [ ]:
# 위험 관련 변수 선택
risk_vars = ['ALT', 'Gtp', 'AST', 'waist(cm)', 'HDL']

# 4개 집단 평균
group_mean = train_clean.groupby('group4')[risk_vars].mean().loc[group_order]

# HDL은 높을수록 좋으므로, 위험 방향으로 뒤집기
group_mean_for_risk = group_mean.copy()
group_mean_for_risk['HDL'] = -group_mean_for_risk['HDL']

# 표준화(z-score)
risk_z = (group_mean_for_risk - group_mean_for_risk.mean()) / group_mean_for_risk.std()

# 그래프
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(risk_vars))
width = 0.18

for i, grp in enumerate(group_order):
    ax.bar(
        x + (i - 1.5) * width,
        risk_z.loc[grp].values,
        width=width,
        label=grp,
        color=group_colors[i]
    )

ax.axhline(0, color=COLOR_TEXT, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(['ALT', 'GTP', 'AST', '허리둘레', 'HDL(역방향)'])
ax.set_title('4개 집단의 위험 프로파일 비교 (표준화 점수)', color=COLOR_TEXT)
ax.set_ylabel('표준화 위험 점수', color=COLOR_TEXT)
ax.grid(axis='y', linestyle='--', color=COLOR_GRID, alpha=0.6)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 저BMI/고BMI별 흡연 효과 계산
interaction_vars = ['ALT', 'Gtp', 'AST', 'waist(cm)']

rows = []
for var in interaction_vars:
    low_non = train_clean[(train_clean['high_BMI'] == 0) & (train_clean['smoking'] == 0)][var].mean()
    low_smk = train_clean[(train_clean['high_BMI'] == 0) & (train_clean['smoking'] == 1)][var].mean()
    high_non = train_clean[(train_clean['high_BMI'] == 1) & (train_clean['smoking'] == 0)][var].mean()
    high_smk = train_clean[(train_clean['high_BMI'] == 1) & (train_clean['smoking'] == 1)][var].mean()

    rows.append([
        var,
        low_smk - low_non,
        high_smk - high_non
    ])

effect_df = pd.DataFrame(rows, columns=['변수', '저BMI에서 흡연 효과', '고BMI에서 흡연 효과'])

# 그래프
x = np.arange(len(effect_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, effect_df['저BMI에서 흡연 효과'], width,
               color=COLOR_LOW_BMI, label='저BMI')
bars2 = ax.bar(x + width/2, effect_df['고BMI에서 흡연 효과'], width,
               color=COLOR_HIGH_BMI, label='고BMI')

ax.set_xticks(x)
ax.set_xticklabels(effect_df['변수'])
ax.axhline(0, color=COLOR_TEXT, linewidth=1)
ax.set_title('BMI 집단별 흡연의 추가 악영향 비교', color=COLOR_TEXT)
ax.set_ylabel('흡연 효과 (흡연자 평균 - 비흡연자 평균)', color=COLOR_TEXT)
ax.grid(axis='y', linestyle='--', color=COLOR_GRID, alpha=0.6)
ax.legend()

for bars in [bars1, bars2]:
    for b in bars:
        y = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, y, f'{y:.2f}', ha='center', va='bottom', color=COLOR_TEXT)

plt.tight_layout()
plt.show()

effect_df

In [ ]:
# 고BMI 집단만 추출
high_bmi_df = train_clean[train_clean['high_BMI'] == 1].copy()

# 주요 변수 평균
target_vars = ['ALT', 'Gtp', 'AST', 'waist(cm)', 'HDL']
high_bmi_mean = high_bmi_df.groupby('smoking')[target_vars].mean().T
high_bmi_mean.columns = ['비흡연', '흡연']

# 그래프
x = np.arange(len(target_vars))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, high_bmi_mean['비흡연'], width, color=COLOR_NONSMOKER, label='비흡연')
bars2 = ax.bar(x + width/2, high_bmi_mean['흡연'], width, color=COLOR_SMOKER, label='흡연')

ax.set_xticks(x)
ax.set_xticklabels(['ALT', 'GTP', 'AST', '허리둘레', 'HDL'])
ax.set_title('고BMI 집단 내부의 비흡연 vs 흡연 비교', color=COLOR_TEXT)
ax.set_ylabel('평균값', color=COLOR_TEXT)
ax.grid(axis='y', linestyle='--', color=COLOR_GRID, alpha=0.6)
ax.legend()

for bars in [bars1, bars2]:
    for b in bars:
        y = b.get_height()
        ax.text(b.get_x() + b.get_width()/2, y, f'{y:.1f}', ha='center', va='bottom', color=COLOR_TEXT, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
risk_df = train_clean.copy()

# 위험 방향 통일
risk_df['HDL_risk'] = -risk_df['HDL']

risk_vars_score = ['ALT', 'Gtp', 'waist(cm)', 'HDL_risk']

# z-score 표준화
for col in risk_vars_score:
    risk_df[col + '_z'] = (risk_df[col] - risk_df[col].mean()) / risk_df[col].std()

risk_df['risk_score'] = risk_df[[c + '_z' for c in risk_vars_score]].mean(axis=1)

risk_score_mean = risk_df.groupby('group4')['risk_score'].mean().loc[group_order]

plt.figure(figsize=(8, 5))
bars = plt.bar(risk_score_mean.index, risk_score_mean.values, color=group_colors)

plt.axhline(0, color=COLOR_TEXT, linewidth=1)
plt.title('4개 집단의 통합 위험지수 비교', color=COLOR_TEXT)
plt.ylabel('평균 위험지수', color=COLOR_TEXT)
plt.grid(axis='y', linestyle='--', color=COLOR_GRID, alpha=0.6)
plt.xticks(rotation=15)

for i, v in enumerate(risk_score_mean.values):
    plt.text(i, v, f'{v:.2f}', ha='center', va='bottom', color=COLOR_TEXT)

plt.tight_layout()
plt.show()